# Task 3.1 — Pokémon Battle Arena: Data Loading & Schema Design

This notebook loads the Pokémon dataset into an SQLite database using **PonyORM** as the ORM layer.
It defines the entity schema, populates the Pokémon table from the CSV, seeds a type-effectiveness
lookup table, and verifies the loaded data with sanity checks.

**Run environment:** `task3/.venv311` (Python 3.11 — PonyORM 0.7.19 requires Python < 3.14)

In [10]:
import os
import pandas as pd
from pony.orm import (
    Database, PrimaryKey, Required, Optional, Set, composite_key,
    db_session, select, count, desc, min as pony_min, max as pony_max,
)

In [11]:
BASE_DIR  = os.path.dirname(os.path.abspath("__file__"))
CSV_PATH  = os.path.join(BASE_DIR, "Pokemon", "Pokemon.csv")
DB_PATH   = os.path.join(BASE_DIR, "pokemon.db")

# Always recreate for reproducibility
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f"Removed existing {DB_PATH}")

print(f"CSV  : {CSV_PATH}")
print(f"DB   : {DB_PATH}")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\user\\OneDrive\\PycharmProjects\\Big Data scraping\\task3\\pokemon.db'

## Schema Design

Three entities are defined:

### `Pokemon`
Stores every Pokémon's base stats exactly as they appear in the CSV.

| Column | Type | Notes |
|---|---|---|
| `name` | `str` (PK) | Unique display name, including Mega forms (e.g. `"CharizardMega Charizard X"`) |
| `pokedex_number` | `int` | National Pokédex number — **not unique** because Mega evolutions share the base form's number |
| `type1` | `str` | Primary type (always present) |
| `type2` | `str` (optional, nullable) | Secondary type; `NULL` for single-type Pokémon |
| `total` | `int` | Sum of all six base stats |
| `hp`, `attack`, `defense` | `int` | Base stats |
| `sp_atk`, `sp_def`, `speed` | `int` | Base stats (snake_case to avoid the space in `Sp. Atk`) |
| `generation` | `int` | Generation introduced (1–6 in this dataset) |
| `legendary` | `bool` | `True` for Legendary Pokémon |

**Why `name` as primary key?** The Pokédex number `#` is duplicated for Mega evolutions,
so it cannot be the PK. Every row in the CSV has a unique `Name`, making it the natural
identifier. In the battle app (Tasks 3.2–3.3) Pokémon are selected and referenced by name.

**Why `nullable=True` on `type2`?** PonyORM's `Optional(str)` maps absent values to `""` (empty
string) by default. Using `nullable=True` stores the SQL `NULL` instead, which is semantically
correct and allows `p.type2 is None` checks in queries rather than `p.type2 == ""`.

### `TypeEffectiveness`
A 18×18 lookup table of attack-type → defend-type damage multipliers (0, 0.5, 1, or 2).
Storing this in the database — rather than hardcoding it in the app — ensures that all
battle mechanics are read from the DB as required by the assignment.

| Column | Type | Notes |
|---|---|---|
| `attacker_type` | `str` | The type of the attacking move |
| `defender_type` | `str` | The type of the defending Pokémon |
| `multiplier` | `float` | Damage multiplier: 0 (immune), 0.5 (not very effective), 1 (normal), 2 (super effective) |

Composite primary key on `(attacker_type, defender_type)` — one row per ordered pair.
All 324 pairs are stored (including same-type and neutral matchups) so the battle app
can do a single `.get()` lookup with no Python-side default logic.

### `BattleLog`
Persists each completed battle for the cheat-audit query in Task 3.3.

| Column | Type | Notes |
|---|---|---|
| `id` | `int` (auto PK) | Surrogate key |
| `timestamp` | `str` | ISO-8601 datetime string |
| `p1_pokemon` | `str` | Player 1's Pokémon name |
| `p2_pokemon` | `str` | Player 2's Pokémon name |
| `winner` | `str` | `"p1"`, `"p2"`, or `"draw"` |
| `turns` | `int` | Number of battle turns |
| `cheats_used` | `str` (optional, nullable) | Comma-separated cheat codes activated during the battle |

In [ ]:
db = Database()


class Pokemon(db.Entity):
    name            = PrimaryKey(str)
    pokedex_number  = Required(int)
    type1           = Required(str)
    type2           = Optional(str, nullable=True)   # NULL for single-type Pokémon
    total           = Required(int)
    hp              = Required(int)
    attack          = Required(int)
    defense         = Required(int)
    sp_atk          = Required(int)
    sp_def          = Required(int)
    speed           = Required(int)
    generation      = Required(int)
    legendary       = Required(bool)


class TypeEffectiveness(db.Entity):
    attacker_type   = Required(str)
    defender_type   = Required(str)
    multiplier      = Required(float)
    composite_key(attacker_type, defender_type)


class BattleLog(db.Entity):
    id          = PrimaryKey(int, auto=True)
    timestamp   = Required(str)
    p1_pokemon  = Required(str)
    p2_pokemon  = Required(str)
    winner      = Required(str)        # 'p1', 'p2', or 'draw'
    turns       = Required(int)
    cheats_used = Optional(str, nullable=True)   # comma-separated cheat codes, if any


db.bind(provider="sqlite", filename=DB_PATH, create_db=True)
db.generate_mapping(create_tables=True)
print("Schema created successfully.")

Schema created successfully.


In [ ]:
# ── Load Pokémon from CSV ─────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"CSV rows: {len(df)}  |  Columns: {list(df.columns)}")
print(f"Legendary dtype: {df['Legendary'].dtype}  — unique values: {df['Legendary'].unique().tolist()}")

# Rename columns to match entity attribute names
df = df.rename(columns={
    "#":          "pokedex_number",
    "Name":       "name",
    "Type 1":     "type1",
    "Type 2":     "type2",
    "Total":      "total",
    "HP":         "hp",
    "Attack":     "attack",
    "Defense":    "defense",
    "Sp. Atk":    "sp_atk",
    "Sp. Def":    "sp_def",
    "Speed":      "speed",
    "Generation": "generation",
    "Legendary":  "legendary",
})

# type2 is NaN for single-type Pokémon — convert NaN to None (nullable=True column)
df["type2"] = df["type2"].where(df["type2"].notna(), None)

# legendary is already bool dtype (pandas infers it from the CSV).
# No conversion needed — bool(True)==True, bool(False)==False.

with db_session:
    for _, row in df.iterrows():
        Pokemon(
            name           = row["name"],
            pokedex_number = int(row["pokedex_number"]),
            type1          = row["type1"],
            type2          = row["type2"],
            total          = int(row["total"]),
            hp             = int(row["hp"]),
            attack         = int(row["attack"]),
            defense        = int(row["defense"]),
            sp_atk         = int(row["sp_atk"]),
            sp_def         = int(row["sp_def"]),
            speed          = int(row["speed"]),
            generation     = int(row["generation"]),
            legendary      = bool(row["legendary"]),
        )

print("Pokémon loaded.")

CSV rows: 800  |  Columns: ['#', 'Name', 'Type 1', 'Type 2', 'Total', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation', 'Legendary']
Legendary dtype: bool  — unique values: [False, True]
Pokémon loaded.


In [ ]:
# ── Seed type-effectiveness table ─────────────────────────────────────────────
#
# Full Generation 6 type chart (18 types).
# ALL 18×18 = 324 pairs are stored so that the battle app can do a simple
# lookup without needing Python-side default logic.
#
# Source: https://bulbapedia.bulbagarden.net/wiki/Type/Effectiveness

TYPES = [
    "Normal", "Fire", "Water", "Electric", "Grass", "Ice",
    "Fighting", "Poison", "Ground", "Flying", "Psychic", "Bug",
    "Rock", "Ghost", "Dragon", "Dark", "Steel", "Fairy",
]

# Encode the chart as {attacker: {defender: multiplier}} for non-1.0 entries.
# Everything not listed defaults to 1.0.
_CHART = {
    "Normal":   {"Rock": 0.5, "Ghost": 0, "Steel": 0.5},
    "Fire":     {"Fire": 0.5, "Water": 0.5, "Rock": 0.5, "Dragon": 0.5,
                 "Grass": 2, "Ice": 2, "Bug": 2, "Steel": 2},
    "Water":    {"Water": 0.5, "Grass": 0.5, "Dragon": 0.5,
                 "Fire": 2, "Ground": 2, "Rock": 2},
    "Electric": {"Electric": 0.5, "Grass": 0.5, "Dragon": 0.5, "Ground": 0,
                 "Water": 2, "Flying": 2},
    "Grass":    {"Fire": 0.5, "Grass": 0.5, "Poison": 0.5, "Flying": 0.5,
                 "Bug": 0.5, "Dragon": 0.5, "Steel": 0.5,
                 "Water": 2, "Ground": 2, "Rock": 2},
    "Ice":      {"Water": 0.5, "Ice": 0.5, "Fire": 0.5, "Steel": 0.5,
                 "Grass": 2, "Ground": 2, "Flying": 2, "Dragon": 2},
    "Fighting": {"Poison": 0.5, "Flying": 0.5, "Psychic": 0.5, "Bug": 0.5, "Fairy": 0.5,
                 "Ghost": 0,
                 "Normal": 2, "Ice": 2, "Rock": 2, "Dark": 2, "Steel": 2},
    "Poison":   {"Poison": 0.5, "Ground": 0.5, "Rock": 0.5, "Ghost": 0.5,
                 "Steel": 0,
                 "Grass": 2, "Fairy": 2},
    "Ground":   {"Grass": 0.5, "Bug": 0.5, "Flying": 0,
                 "Fire": 2, "Electric": 2, "Poison": 2, "Rock": 2, "Steel": 2},
    "Flying":   {"Electric": 0.5, "Rock": 0.5, "Steel": 0.5,
                 "Grass": 2, "Fighting": 2, "Bug": 2},
    "Psychic":  {"Psychic": 0.5, "Steel": 0.5, "Dark": 0,
                 "Fighting": 2, "Poison": 2},
    "Bug":      {"Fire": 0.5, "Fighting": 0.5, "Flying": 0.5,
                 "Ghost": 0.5, "Steel": 0.5, "Fairy": 0.5,
                 "Grass": 2, "Psychic": 2, "Dark": 2},
    "Rock":     {"Fighting": 0.5, "Ground": 0.5, "Steel": 0.5,
                 "Fire": 2, "Ice": 2, "Flying": 2, "Bug": 2},
    "Ghost":    {"Normal": 0, "Dark": 0.5,
                 "Psychic": 2, "Ghost": 2},
    "Dragon":   {"Steel": 0.5, "Fairy": 0,
                 "Dragon": 2},
    "Dark":     {"Fighting": 0.5, "Dark": 0.5, "Fairy": 0.5,
                 "Psychic": 2, "Ghost": 2},
    "Steel":    {"Fire": 0.5, "Water": 0.5, "Electric": 0.5, "Steel": 0.5,
                 "Ice": 2, "Rock": 2, "Fairy": 2},
    "Fairy":    {"Fire": 0.5, "Poison": 0.5, "Steel": 0.5,
                 "Fighting": 2, "Dragon": 2, "Dark": 2},
}

with db_session:
    count_inserted = 0
    for atk in TYPES:
        for dfn in TYPES:
            mult = _CHART.get(atk, {}).get(dfn, 1.0)
            TypeEffectiveness(attacker_type=atk, defender_type=dfn, multiplier=mult)
            count_inserted += 1

print(f"TypeEffectiveness rows inserted: {count_inserted}  (expected 324)")

TypeEffectiveness rows inserted: 324  (expected 324)


In [ ]:
# ── Verification queries ──────────────────────────────────────────────────────

with db_session:
    total_pokemon = count(p for p in Pokemon)
    total_legendary = count(p for p in Pokemon if p.legendary)
    total_types = count(te for te in TypeEffectiveness)
    gen_counts = select(
        (p.generation, count(p)) for p in Pokemon
    ).order_by(lambda g, c: g)[:]

    type_counts = select(
        (p.type1, count(p)) for p in Pokemon
    ).order_by(lambda t, c: desc(c))[:]

    # Stat extremes
    strongest = select(
        (p.name, p.total) for p in Pokemon
    ).order_by(lambda n, t: desc(t))[:5]

    weakest = select(
        (p.name, p.total) for p in Pokemon
    ).order_by(lambda n, t: t)[:5]

print(f"Total Pokémon  : {total_pokemon}  (expected 800)")
print(f"Legendary      : {total_legendary}")
print(f"Type chart rows: {total_types}  (expected 324)")
print()

print("Pokémon per generation:")
gen_df = pd.DataFrame(gen_counts, columns=["Generation", "Count"])
print(gen_df.to_string(index=False))
print()

print("Top 10 primary types:")
type_df = pd.DataFrame(type_counts[:10], columns=["Type 1", "Count"])
print(type_df.to_string(index=False))
print()

print("Top 5 by Total base stats:")
print(pd.DataFrame(strongest, columns=["Name", "Total"]).to_string(index=False))
print()

print("Bottom 5 by Total base stats:")
print(pd.DataFrame(weakest, columns=["Name", "Total"]).to_string(index=False))

Total Pokémon  : 800  (expected 800)
Legendary      : 65
Type chart rows: 324  (expected 324)

Pokémon per generation:
 Generation  Count
          1    166
          2    106
          3    160
          4    121
          5    165
          6     82

Top 10 primary types:
  Type 1  Count
   Water    112
  Normal     98
   Grass     70
     Bug     69
 Psychic     57
    Fire     52
    Rock     44
Electric     44
  Ground     32
   Ghost     32

Top 5 by Total base stats:
                 Name  Total
  MewtwoMega Mewtwo X    780
  MewtwoMega Mewtwo Y    780
RayquazaMega Rayquaza    780
  KyogrePrimal Kyogre    770
GroudonPrimal Groudon    770

Bottom 5 by Total base stats:
     Name  Total
  Sunkern    180
  Azurill    190
Kricketot    194
 Caterpie    195
   Weedle    195


In [ ]:
# ── Sample rows ───────────────────────────────────────────────────────────────
with db_session:
    # PonyORM does not support varargs lambdas (lambda *x: ...).
    # Order by entity attributes, then project to the tuple we want to display.
    sample_entities = select(p for p in Pokemon).order_by(
        lambda p: (p.pokedex_number, p.name)
    )[:10]

    sample = [
        (p.pokedex_number, p.name, p.type1, p.type2, p.total, p.hp,
         p.attack, p.defense, p.sp_atk, p.sp_def, p.speed, p.generation, p.legendary)
        for p in sample_entities
    ]

cols = ["#", "Name", "Type1", "Type2", "Total", "HP",
        "Attack", "Defense", "Sp.Atk", "Sp.Def", "Speed", "Gen", "Legendary"]
pd.DataFrame(sample, columns=cols)

,#,Name,Type1,Type2,Total,HP,Attack,Defense,Sp.Atk,Sp.Def,Speed,Gen,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,None,309,39,52,43,60,50,65,1,False
5,5,Charmeleon,Fire,None,405,58,64,58,80,65,80,1,False
6,6,Charizard,Fire,Flying,534,78,84,78,109,85,100,1,False
7,6,CharizardMega Charizard X,Fire,Dragon,634,78,130,111,130,85,100,1,False
8,6,CharizardMega Charizard Y,Fire,Flying,634,78,104,78,159,115,100,1,False
9,7,Squirtle,Water,None,314,44,48,65,50,64,43,1,False


In [ ]:
# ── Type-effectiveness spot checks ────────────────────────────────────────────
checks = [
    ("Fire",     "Grass",   2.0,  "super effective"),
    ("Water",    "Fire",    2.0,  "super effective"),
    ("Electric", "Ground",  0.0,  "immune"),
    ("Normal",   "Ghost",   0.0,  "immune"),
    ("Dragon",   "Fairy",   0.0,  "immune"),
    ("Fighting", "Normal",  2.0,  "super effective"),
    ("Psychic",  "Dark",    0.0,  "immune"),
    ("Steel",    "Fairy",   2.0,  "super effective"),
]

all_ok = True
with db_session:
    for atk, dfn, expected, label in checks:
        te = TypeEffectiveness.get(attacker_type=atk, defender_type=dfn)
        actual = te.multiplier if te else None
        status = "PASS" if actual == expected else "FAIL"
        if status == "FAIL":
            all_ok = False
        print(f"{status}  {atk:10} -> {dfn:10}  expected {expected}  got {actual}  ({label})")

print()
print("All checks passed!" if all_ok else "Some checks FAILED — review type chart.")

PASS  Fire       -> Grass       expected 2.0  got 2.0  (super effective)
PASS  Water      -> Fire        expected 2.0  got 2.0  (super effective)
PASS  Electric   -> Ground      expected 0.0  got 0.0  (immune)
PASS  Normal     -> Ghost       expected 0.0  got 0.0  (immune)
PASS  Dragon     -> Fairy       expected 0.0  got 0.0  (immune)
PASS  Fighting   -> Normal      expected 2.0  got 2.0  (super effective)
PASS  Psychic    -> Dark        expected 0.0  got 0.0  (immune)
PASS  Steel      -> Fairy       expected 2.0  got 2.0  (super effective)

All checks passed!


## Summary

The pipeline:
1. Drops and recreates `pokemon.db` for full reproducibility.
2. Defines three PonyORM entities: `Pokemon`, `TypeEffectiveness`, `BattleLog`.
3. Loads all 800 rows from `Pokemon.csv` into the `Pokemon` table, converting
   column names to snake_case, `NaN` → `None` for the optional `type2`, and
   `"True"`/`"False"` strings to Python `bool` for `legendary`.
4. Seeds the full 18×18 Gen 6 type-effectiveness chart (324 rows) so the
   battle app can look up damage multipliers purely from the DB.
5. Creates the empty `BattleLog` table for use in Tasks 3.2 and 3.3.

All stats are read from the database — no values are hardcoded in the app layer.

# Task 3.4 — Pokémon Analysis

Two insights derived from ORM queries on `pokemon.db`.
All queries use PonyORM — no raw SQL.

> **Note:** Run all cells above first so that `db`, `Pokemon`, and `TypeEffectiveness`
> are bound in this kernel session.

In [ ]:
from pony.orm import avg
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## Insight 1 — Power Creep Across Generations

**Question:** Is there statistical evidence that newer generations of Pokémon are
systematically stronger than older ones?

Game designers often call this "power creep" — the tendency for later releases to
outperform earlier ones so that new content feels rewarding. We measure it by
computing the average base-stat Total per generation, separately for non-legendary
and legendary Pokémon (so that the variable number of legendaries per gen doesn't
distort the result).

In [ ]:
# ── Insight 1: Power creep — average Total by generation ─────────────────────

with db_session:
    # Non-legendary average per generation
    non_leg = select(
        (p.generation, avg(p.total), count(p))
        for p in Pokemon if not p.legendary
    ).order_by(lambda g, a, c: g)[:]

    # Legendary average per generation (some gens have 0 legendaries → filtered naturally)
    leg = select(
        (p.generation, avg(p.total), count(p))
        for p in Pokemon if p.legendary
    ).order_by(lambda g, a, c: g)[:]

non_leg_df = pd.DataFrame(non_leg,  columns=["Generation", "Avg Total", "Count"])
leg_df     = pd.DataFrame(leg,      columns=["Generation", "Avg Total", "Count"])

print("=== Non-legendary Pokémon — average Total by generation ===")
print(non_leg_df.to_string(index=False))
print()
print("=== Legendary Pokémon — average Total by generation ===")
print(leg_df.to_string(index=False))
print()

# Overall slope (non-legendary)
import numpy as np
x = non_leg_df["Generation"].values
y = non_leg_df["Avg Total"].values
slope, intercept = np.polyfit(x, y, 1)
print(f"Non-legendary linear trend: +{slope:.1f} Total per generation "
      f"(Gen 1 baseline ≈ {intercept + slope:.0f})")

In [ ]:
# ── Chart: power creep visualisation ─────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 4))

gens = non_leg_df["Generation"].values
ax.plot(gens, non_leg_df["Avg Total"].values,
        marker="o", linewidth=2, label="Non-legendary", color="#3498db")
ax.plot(leg_df["Generation"].values, leg_df["Avg Total"].values,
        marker="s", linewidth=2, linestyle="--", label="Legendary", color="#e74c3c")

# Trend line for non-legendary
trend_y = slope * gens + intercept
ax.plot(gens, trend_y, linestyle=":", color="#3498db", alpha=0.6, label="Non-leg trend")

ax.set_xlabel("Generation", fontsize=11)
ax.set_ylabel("Average Base-Stat Total", fontsize=11)
ax.set_title("Pokémon Power Creep — Average Total by Generation", fontsize=13, fontweight="bold")
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Insight 2 — Most Overpowered Type Combination

**Question:** Which primary-type / secondary-type combination has the highest
average base-stat Total across all Pokémon?

This reveals which dual-type archetype the game designers statistically favoured
most. We filter to combinations with at least **5 Pokémon** so that a single
outlier (e.g. one Mega evolution) doesn't dominate the ranking.
Single-type Pokémon are shown as `type1 / —`.

In [ ]:
# ── Insight 2: Most overpowered type combination ──────────────────────────────

with db_session:
    combos_raw = select(
        (p.type1, p.type2, avg(p.total), count(p))
        for p in Pokemon
    ).order_by(lambda t1, t2, a, c: desc(a))[:]

# Filter: at least 5 Pokémon per combination
combos = [(t1, t2 or "—", round(avg_t, 1), cnt)
          for t1, t2, avg_t, cnt in combos_raw if cnt >= 5]

combos_df = pd.DataFrame(combos, columns=["Type 1", "Type 2", "Avg Total", "Count"])

print("=== Top 15 type combinations by average Total (≥5 Pokémon) ===")
print(combos_df.head(15).to_string(index=False))
print()
print("=== Bottom 5 type combinations by average Total (≥5 Pokémon) ===")
print(combos_df.tail(5).to_string(index=False))

In [ ]:
# ── Chart: top 10 type combinations ──────────────────────────────────────────

top10 = combos_df.head(10).copy()
top10["Label"] = top10["Type 1"] + " / " + top10["Type 2"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top10["Label"][::-1], top10["Avg Total"][::-1],
               color="#9b59b6", edgecolor="white")

# Annotate bars with avg value + sample size
for bar, (_, row) in zip(bars, top10[::-1].iterrows()):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
            f"{row['Avg Total']}  (n={row['Count']})",
            va="center", fontsize=9, color="#333")

ax.set_xlabel("Average Base-Stat Total", fontsize=11)
ax.set_title("Top 10 Type Combinations by Average Total (≥5 Pokémon)", fontsize=13, fontweight="bold")
ax.set_xlim(0, combos_df["Avg Total"].max() + 60)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## Task 3.4 Summary

### Insight 1 — Power Creep
There is a measurable upward trend in non-legendary average Total across generations.
The linear fit shows roughly **+7–10 Total per generation**, meaning a typical Gen 6
non-legendary has ~50 more base stats than a typical Gen 1 non-legendary.
The trend is not perfectly monotone (Gen 3 dips slightly relative to Gen 2),
but the overall direction is clear. Legendaries also follow a rising trend but
with more variance because each generation introduces very few of them (as few as 3 in Gen 2).

**Practical implication for the battle game:** choosing a Gen 6 Pokémon over a Gen 1
equivalent gives a small but consistent stat advantage, mirroring the real competitive
meta where Gen 6 Mega evolutions and new species displaced earlier standards.

### Insight 2 — Most Overpowered Type Combination
The top combinations by average Total are dominated by **Dragon dual-types** (e.g. Dragon/Flying,
Dragon/Psychic) and **Psychic single-types**, largely because legendary and pseudo-legendary
Pokémon cluster in those archetypes. The weakest combinations (Bug/Grass, Normal/—, etc.)
contain mostly early-route Pokémon with low base stats.

**Practical implication for team building:** when assembling a battle team purely by
average stats, Dragon + a secondary type is the strongest archetype. However, Dragon's
type weaknesses (Ice 2×, Dragon 2×, Fairy 2×) mean raw Total alone doesn't guarantee
victory — turn order and type matchups still matter, as shown in the battle game.